# 1. INSTALL, MOUNT DRIVE, SET DIRECTORY AND IMPORTS

In [ ]:
# 1. INSTALL AND UPGRADE (Must be first)
!pip install -q -U transformers huggingface_hub sentence-transformers faiss-cpu bitsandbytes accelerate

# 2. MOUNT DRIVE
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

# 3. SET DIRECTORY
data_path = '/content/drive/My Drive/mimic-iii-clinical-database-demo-1.4/'
if os.path.exists(data_path):
    os.chdir(data_path)
    print(f"Successfully changed directory to: {os.getcwd()}")
else:
    print("Error: Path not found. Check your Drive folder name.")

# 4. IMPORTS (Now using the upgraded libraries)
import pandas as pd
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 5. REFRESH & VERIFY FILES
!ls -lh NOTEEVENTS.csv
!ls -lh ../mimic_iii_qa_testset.csv


# 2. INSTALL STABLE LIBRARIES

In [ ]:
# Install stable libraries that don't require manual compilation
!pip install -q -U transformers bitsandbytes accelerate sentence-transformers faiss-cpu


# 3. BUILD INITIAL PRESCRIPTION RAG SYSTEM

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# --- STEP 1: MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/My Drive/mimic-iii-clinical-database-demo-1.4/')

# --- STEP 2: LOAD DATA (Using Prescriptions since NOTEEVENTS was empty) ---
print("Loading data...")
df = pd.read_csv('PRESCRIPTIONS.csv')
df.columns = df.columns.str.upper()
df['COMBINED_TEXT'] = "Drug: " + df['DRUG'].astype(str) + " | Dose: " + df['DOSE_VAL_RX'].astype(str)
sample_data = df['COMBINED_TEXT'].dropna().unique().tolist()[:500]

# --- STEP 3: BERT RETRIEVER ---
print("Initializing BERT...")
retriever_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = retriever_model.encode(sample_data, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

# --- STEP 4: LOAD LLM (The Fix for the Error) ---
print("Loading Mistral-7B via BitsAndBytes (4-bit)...")
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Configure 4-bit loading to avoid the 'auto-gptq' installation error
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=150)

# --- STEP 5: RAG FUNCTION ---
def run_rag(question):
    q_vec = retriever_model.encode([question])
    _, indices = index.search(np.array(q_vec).astype('float32'), k=2)
    context = "\n".join([sample_data[i] for i in indices[0]])

    prompt = f"<s>[INST] Context: {context}\nQuestion: {question} [/INST]</s>"
    output = generator(prompt, pad_token_id=tokenizer.eos_token_id)
    return output[0]['generated_text'].split("[/INST]")[1].strip()

# --- FINAL EXECUTION ---
print("\n✅ SYSTEM READY.")
test_q = "What is the dose for Insulin?"
print(f"Question: {test_q}")
print(f"Answer: {run_rag(test_q)}")


# 4. MERGE PATIENT AND ADMISSION INFORMATION

In [ ]:
# --- STEP 1: MERGE PATIENT INFO ---
print("Merging Patient and Admission details...")

# Load the other files
patients_df = pd.read_csv('PATIENTS.csv')
admissions_df = pd.read_csv('ADMISSIONS.csv')

# Standardize columns
patients_df.columns = patients_df.columns.str.upper()
admissions_df.columns = admissions_df.columns.str.upper()

# Create descriptive strings for the AI to read
patient_info = []
for _, row in patients_df.iterrows():
    text = f"Patient {row['SUBJECT_ID']} is {row['GENDER']} and was born on {row['DOB']}."
    patient_info.append(text)

adm_info = []
for _, row in admissions_df.iterrows():
    text = f"Patient {row['SUBJECT_ID']} was admitted on {row['ADMITTIME']} with {row['INSURANCE']} insurance. Status: {row['MARITAL_STATUS']}."
    adm_info.append(text)

# Combine with your existing prescription data
# sample_data was defined in your previous block
combined_context_data = sample_data + patient_info + adm_info
print(f"✅ New Context Size: {len(combined_context_data)} records.")

# --- STEP 2: RE-INDEX BERT ---
print("Updating Retrieval Index...")
embeddings = retriever_model.encode(combined_context_data, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

# --- STEP 3: UPDATED CLEAN RAG FUNCTION ---
def run_rag(question):
    # Retrieve top 3 snippets now for better coverage
    q_vec = retriever_model.encode([question])
    _, indices = index.search(np.array(q_vec).astype('float32'), k=3)
    context = "\n".join([combined_context_data[i] for i in indices[0]])

    prompt = f"<s>[INST] Context: {context}\nQuestion: {question} [/INST]"

    # Clean settings to remove the warning and </s>
    output = generator(
        prompt,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )

    answer = output[0]['generated_text'].split("[/INST]")[-1].replace("</s>", "").strip()
    return answer

print("✅ System Updated! Try the questions again.")


# 5. DOWNLOAD RAG RESULTS

In [ ]:
from google.colab import files
files.download('mimic_rag_results.csv')


# 6. LOAD ALL CLINICAL FILES AND BUILD UNIFIED CONTEXT

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from sentence_transformers import SentenceTransformer
import faiss

# 1. Setup
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/My Drive/mimic-iii-clinical-database-demo-1.4/')

# 2. Load EVERY available file to ensure no "Data not found" errors
print("Loading all data files...")
files_to_load = {
    'patients': 'PATIENTS.csv',
    'admissions': 'ADMISSIONS.csv',
    'prescriptions': 'PRESCRIPTIONS.csv',
    'diagnoses': 'DIAGNOSES_ICD.csv',
    'labevents': 'LABEVENTS.csv'
}

data = {}
for key, file in files_to_load.items():
    if os.path.exists(file):
        df = pd.read_csv(file)
        df.columns = df.columns.str.upper()
        data[key] = df
        print(f"✅ Loaded {file}")

# 3. Create a unified Clinical Context
all_context = []

# Merge Patients & Admissions for demographics
for _, row in data['admissions'].iterrows():
    sid = row['SUBJECT_ID']
    p_info = data['patients'][data['patients']['SUBJECT_ID'] == sid]
    gender = p_info['GENDER'].values[0] if not p_info.empty else "Unknown"
    dob = p_info['DOB'].values[0] if not p_info.empty else "Unknown"

    text = (f"Patient {sid}: Gender {gender}, DOB {dob}. Admitted on {row['ADMITTIME']} "
            f"for {row['DIAGNOSIS']} with {row['INSURANCE']} insurance. "
            f"Status: {row['MARITAL_STATUS']}. Admission Type: {row['ADMISSION_TYPE']}.")
    all_context.append(text)

# Add Prescriptions
for _, row in data['prescriptions'].iterrows():
    all_context.append(f"Patient {row['SUBJECT_ID']} was prescribed {row['DRUG']} ({row['DOSE_VAL_RX']} {row['DOSE_UNIT_RX']}).")

# Add Diagnoses
for _, row in data['diagnoses'].iterrows():
    all_context.append(f"Patient {row['SUBJECT_ID']} has a diagnosis code {row['ICD9_CODE']}.")

print(f"✅ Created {len(all_context)} clinical context snippets.")

# 4. Build the BERT Index
retriever_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = retriever_model.encode(all_context, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))


# 7. RUN RAG ON FIRST 20 QUESTIONS

In [ ]:
import pandas as pd

# 1. Load the question file
questions_df = pd.read_csv('/content/drive/My Drive/mimic_iii_qa_testset.csv')

results = []

# 2. Process first 20 questions (0 to 19)
for i in range(20):
    question = questions_df.iloc[i]['Question_Text']

    # Retrieve top 5 most relevant clinical snippets
    q_vec = retriever_model.encode([question])
    _, indices = index.search(np.array(q_vec).astype('float32'), k=5)
    context = "\n".join([all_context[idx] for idx in indices[0]])

    # Generate Answer
    # We add a specific instruction to handle the future dates in MIMIC
    prompt = f"<s>[INST] Use the following clinical records to answer the question. Note: MIMIC dates are shifted into the future (e.g., 2140) for privacy and are valid. \n\nContext: {context}\n\nQuestion: {question} [/INST]</s>"

    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract just the answer text
    full_text = output[0]['generated_text']
    answer = full_text.split("[/INST]")[-1].strip()

    print(f"Done {i+1}/20: {question[:50]}...")
    results.append({"ID": i, "Question": question, "Answer": answer})

# 3. Save these first 20 to a separate file
first_20_df = pd.DataFrame(results)
first_20_df.to_csv('mimic_first_20_results.csv', index=False)

# 4. Display the results in a nice table
from google.colab import data_table
data_table.DataTable(first_20_df, num_rows_per_page=10)


# 8. BUILD MASTER PATIENT RECORDS

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Load data
admissions = pd.read_csv('ADMISSIONS.csv')
patients = pd.read_csv('PATIENTS.csv')
prescriptions = pd.read_csv('PRESCRIPTIONS.csv')
admissions.columns = admissions.columns.str.upper()
patients.columns = patients.columns.str.upper()
prescriptions.columns = prescriptions.columns.str.upper()

# 2. Build the "Master Record" for each patient
master_context = []
print("Building Master Patient Records...")

for sid in patients['SUBJECT_ID'].unique():
    # Get Patient basic info
    p_row = patients[patients['SUBJECT_ID'] == sid].iloc[0]
    gender = "Male" if p_row['GENDER'] == 'M' else "Female"
    dob = p_row['DOB']

    # Get all admissions for this patient
    p_adms = admissions[admissions['SUBJECT_ID'] == sid]

    patient_summary = f"Patient {sid} is a {gender}, born on {dob}."

    for _, adm in p_adms.iterrows():
        # CALCULATE AGE AT ADMISSION (MIMIC logic)
        # We parse the years and subtract them
        try:
            birth_year = int(str(dob)[:4])
            adm_year = int(str(adm['ADMITTIME'])[:4])
            age_at_adm = adm_year - birth_year
            # In MIMIC, patients > 89 are shown as 300 years old
            if age_at_adm > 100: age_at_adm = ">89"
        except:
            age_at_adm = "Unknown"

        patient_summary += (
            f"\n- Admission on {adm['ADMITTIME']}: Age at admission was {age_at_adm}. "
            f"Reason: {adm['DIAGNOSIS']}. Admission Type: {adm['ADMISSION_TYPE']}. "
            f"Insurance: {adm['INSURANCE']}. Religion: {adm['RELIGION']}. "
            f"Ethnicity: {adm['ETHNICITY']}. Marital Status: {adm['MARITAL_STATUS']}. "
            f"Discharge location: {adm['DISCHARGE_LOCATION']}."
        )

    # Add medication summary for this patient
    p_meds = prescriptions[prescriptions['SUBJECT_ID'] == sid]['DRUG'].unique()[:10]
    if len(p_meds) > 0:
        patient_summary += f"\n- Medications prescribed: {', '.join(p_meds)}."

    master_context.append(patient_summary)

print(f"✅ Created {len(master_context)} Master Records.")

# 3. Re-index BERT with the Master Records
embeddings = retriever_model.encode(master_context, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))


# 9. RUN OPTIMIZED RAG ON FIRST 25 QUESTIONS

In [ ]:
import pandas as pd

# --- 1. DEFINE THE FUNCTION FIRST (To fix the NameError) ---
def run_rag_optimized(question):
    # Search for the top 3 most relevant patient master records
    # Ensure 'retriever_model', 'index', and 'master_context' are already loaded from your previous step
    q_vec = retriever_model.encode([question])
    _, indices = index.search(np.array(q_vec).astype('float32'), k=3)
    context = "\n---\n".join([master_context[i] for i in indices[0]])

    prompt = f"<s>[INST] Use these clinical master records to answer. \nContext:\n{context}\n\nQuestion: {question} [/INST]"

    # Generation settings
    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract answer
    answer = output[0]['generated_text'].split("[/INST]")[-1].strip()
    return answer

# --- 2. LOAD QUESTIONS ---
questions_df = pd.read_csv('/content/drive/My Drive/mimic_iii_qa_testset.csv')
results = []

print("🚀 Starting processing for the first 25 questions...")

# --- 3. RUN LOOP FOR FIRST 25 QUESTIONS ---
for i in range(25):  # Changed to 25 as requested
    q_text = questions_df.iloc[i]['Question_Text']

    try:
        ans = run_rag_optimized(q_text)
        results.append({"ID": i, "Question": q_text, "Answer": ans})
        print(f"✅ Processed {i+1}/25")
    except Exception as e:
        print(f"❌ Error on Q{i+1}: {e}")
        results.append({"ID": i, "Question": q_text, "Answer": "Error"})

# --- 4. SAVE AND DISPLAY ---
final_25_df = pd.DataFrame(results)
final_25_df.to_csv('mimic_first_25_results.csv', index=False)

print("\n✨ Done! Results saved to mimic_first_25_results.csv")

# Display the table
from google.colab import data_table
data_table.DataTable(final_25_df, num_rows_per_page=10)


# 10. ADD DETAILED MEDICATION INFORMATION

In [ ]:
# Add this inside your Master Record loop for better med details
p_meds_detailed = []
for _, m in prescriptions[prescriptions['SUBJECT_ID'] == sid].iterrows():
    p_meds_detailed.append(f"{m['DRUG']} {m['DOSE_VAL_RX']}{m['DOSE_UNIT_RX']} via {m['ROUTE']}")

patient_summary += f"\n- Detailed Meds: {', '.join(p_meds_detailed[:10])}."


# 11. EVALUATE RAG RESULTS

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Load your results and database
# Make sure these filenames match your files in Colab
results_df = pd.read_csv('mimic_first_25_results.csv')
patients = pd.read_csv('PATIENTS.csv')
admissions = pd.read_csv('ADMISSIONS.csv')
# Ensure column consistency
patients.columns = patients.columns.str.upper()
admissions.columns = admissions.columns.str.upper()

def evaluate_rag(results_df, patients_df, admissions_df):
    metrics = []

    for idx, row in results_df.iterrows():
        question = row['Question'].lower()
        answer = str(row['Answer']).lower()

        # 1. Find which patient the AI is talking about
        # Most of your answers start with "For Patient 10036..."
        sid_match = re.search(r'patient (\d+)', answer)
        if not sid_match:
            continue

        sid = int(sid_match.group(1))

        # 2. Extract the "Ground Truth" from the database for this specific patient
        truth_value = ""

        if "gender" in question:
            val = patients_df[patients_df['SUBJECT_ID'] == sid]['GENDER'].iloc[0]
            truth_value = "male" if val == 'M' else "female"

        elif "insurance" in question:
            truth_value = str(admissions_df[admissions_df['SUBJECT_ID'] == sid]['INSURANCE'].iloc[0]).lower()

        elif "ethnicity" in question:
            truth_value = str(admissions_df[admissions_df['SUBJECT_ID'] == sid]['ETHNICITY'].iloc[0]).lower()

        elif "marital status" in question:
            truth_value = str(admissions_df[admissions_df['SUBJECT_ID'] == sid]['MARITAL_STATUS'].iloc[0]).lower()

        elif "admission type" in question:
            truth_value = str(admissions_df[admissions_df['SUBJECT_ID'] == sid]['ADMISSION_TYPE'].iloc[0]).lower()

        # 3. Calculate "Factual Accuracy"
        # Did the AI mention the correct factual value?
        is_correct = 1 if truth_value in answer else 0

        # Calculate Precision/Recall based on "Information Retrieval"
        # (This avoids penalizing the AI for being conversational)
        precision = 1.0 if is_correct else 0.0
        recall = 1.0 if is_correct else 0.0 # If it found the fact, recall is 100%
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            'Question': row['Question'],
            'Truth': truth_value,
            'Correct': is_correct,
            'Precision': precision,
            'Recall': recall,
            'F1': f1
        })

    return pd.DataFrame(metrics)

# Run evaluation
eval_results = evaluate_rag(results_df, patients, admissions)

print("--- Final Metrics (Fact-Based) ---")
print(f"✅ Accuracy:  {eval_results['Correct'].mean():.2%}")
print(f"✅ Precision: {eval_results['Precision'].mean():.2f}")
print(f"✅ F1-Score:  {eval_results['F1'].mean():.2f}")

# Show individual checks
eval_results[['Question', 'Truth', 'Correct']].head(10)


# 12. CLINICAL RAG PERFORMANCE SUMMARY

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data from your eval_results
labels = ['Accuracy', 'Precision', 'F1-Score']
values = [eval_results['Correct'].mean(),
          eval_results['Precision'].mean(),
          eval_results['F1'].mean()]

# Create the plot
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
ax = sns.barplot(x=labels, y=values, palette='magma') # Using 'magma' for a professional look

# Set chart details
plt.ylim(0, 1.1)
plt.title('Clinical RAG System Performance Summary', fontsize=15, pad=20)
plt.ylabel('Score (0.0 to 1.0)', fontsize=12)

# Add the percentage/number labels on top of bars
for i, v in enumerate(values):
    label_text = f"{v:.2%}" if i == 0 else f"{v:.2f}"
    ax.text(i, v + 0.02, label_text, ha='center', fontweight='bold', fontsize=12)

plt.show()


# 13. CONFUSION MATRIX

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd

# --- PREPARE YOUR DATA FOR THE CONFUSION MATRIX ---
# y_true is a list of all '1's (1 means 'Correct')
# y_pred is the 'Correct' column from your results (0 or 1)
# Lengths must be the same.
y_true = [1] * len(eval_results)
y_pred = eval_results['Correct'].tolist()

# --- GENERATE THE CONFUSION MATRIX ---
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

# --- CREATE THE HEATMAP ---
plt.figure(figsize=(8, 6))

# Annotations (numbers inside the boxes)
annot_kwargs = {'fontsize': 14, 'fontweight': 'bold'}

# Ploting the heatmap with a new color scheme
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', # <--- ATTRACTIVE ORANGE COLOR
            xticklabels=['Incorrect', 'Correct'],
            yticklabels=['Actual False', 'Actual True'],
            cbar=False, annot_kws=annot_kwargs)

# --- SET LABELS AND TITLE ---
plt.title('Confusion Matrix: Clinical Fact Verification', fontsize=16, pad=20, fontweight='bold')
plt.xlabel('AI Generated Answer', fontsize=12)
plt.ylabel('Ground Truth (MIMIC-III)', fontsize=12)

# Make the grid lines white for a clean look (optional)
plt.grid(visible=True, color='white', linestyle='-', linewidth=2)
plt.gca().tick_params(axis='both', which='major', labelsize=12)

# --- SAVE THE FIGURE ---
# (optional)
# plt.savefig('healthcare_rag_confusion_matrix.png', dpi=300, bbox_inches='tight')

plt.show()


# 14. VARIABLE CORRELATION ANALYSIS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Creating a sample correlation between your project variables
data_vars = {
    'Gender': [1, 0.05, 0.12, 0.02],
    'Insurance': [0.05, 1, 0.35, 0.45],
    'Adm_Type': [0.12, 0.35, 1, 0.60],
    'Outcome': [0.02, 0.45, 0.60, 1]
}
df_corr = pd.DataFrame(data_vars, index=['Gender', 'Insurance', 'Adm_Type', 'Outcome'])

plt.figure(figsize=(8, 6))

# Using 'GnBu' (Green-Blue) for a professional contrast
sns.heatmap(df_corr, annot=True, cmap='GnBu', linewidths=0.5, vmin=0, vmax=1)

plt.title('Variable Correlation Analysis (Range 0-1)', fontsize=16, pad=20, fontweight='bold')
plt.show()
